Event Based Narrative Analysis in Literary Texts using NLP

In [ ]:
import spacy
import re
import numpy as np
from collections import Counter, defaultdict
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans
from transformers import pipeline


In [2]:
nlp = spacy.load("en_core_web_sm")

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

embedder = pipeline(
    "feature-extraction",
    model="sentence-transformers/all-MiniLM-L6-v2"
)


Device set to use cuda:0
Device set to use cuda:0


In [3]:
def load_book(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

text = load_book("alice_clean.txt")


In [4]:
def split_into_chapters(text):
    chapters = re.split(r'(CHAPTER [IVXLCDM]+\.?.*)', text, flags=re.IGNORECASE)

    chapter_list = []
    current_title = ""
    current_text = ""

    for i, chunk in enumerate(chapters):
        chunk = chunk.strip()
        if chunk.upper().startswith("CHAPTER"):
            if current_text:
                chapter_list.append({
                    "title": current_title.strip(),
                    "text": current_text.strip()
                })
            current_title = chunk
            current_text = ""
        else:
            current_text += chunk + "\n"

    if current_text:
        chapter_list.append({
            "title": current_title.strip(),
            "text": current_text.strip()
        })

    return chapter_list

chapters = split_into_chapters(text)
print("Chapters detected:", len(chapters))
for c in chapters[:3]:
    print(c["title"])


Chapters detected: 13

CHAPTER I.Down the Rabbit-Hole
CHAPTER II.The Pool of Tears


In [5]:
chapter_sentences = []
global_sent_id = 0

for chap_id, chap in enumerate(chapters):
    doc = nlp(chap["text"])
    for sent in doc.sents:
        chapter_sentences.append({
            "chapter_id": chap_id,
            "chapter_title": chap["title"],
            "global_sent_id": global_sent_id,
            "sentence": sent.text.strip()
        })
        global_sent_id += 1

print("Total sentences across chapters:", len(chapter_sentences))
print(chapter_sentences[:5])


Total sentences across chapters: 1348
[{'chapter_id': 1, 'chapter_title': 'CHAPTER I.Down the Rabbit-Hole', 'global_sent_id': 0, 'sentence': 'Alice was beginning to get very tired of sitting by her sister on the\nbank, and of having nothing to do: once or twice she had peeped into\nthe book her sister was reading, but it had no pictures or\nconversations in it, “and what is the use of a book,” thought Alice\n“without pictures or conversations?”'}, {'chapter_id': 1, 'chapter_title': 'CHAPTER I.Down the Rabbit-Hole', 'global_sent_id': 1, 'sentence': 'So she was considering in her own mind (as well as she could, for the\nhot day made her feel very sleepy and stupid), whether the pleasure of\nmaking a daisy-chain would be worth the trouble of getting up and\npicking the daisies, when suddenly a White Rabbit with pink eyes ran\nclose by her.'}, {'chapter_id': 1, 'chapter_title': 'CHAPTER I.Down the Rabbit-Hole', 'global_sent_id': 2, 'sentence': 'There was nothing so _very_ remarkable in tha

In [6]:
BAD_TRIGGERS = {
    "be", "seem", "have", "occur", "try", "like",
    "think", "feel", "know", "appear"
}

WEAK_TRIGGERS = {
    "begin", "consider", "wonder", "getting", "hope", "wish", "’m", "’s"
}

def extract_events(chapter_sentences):
    events = []
    for cs in chapter_sentences:
        sent_id = cs['global_sent_id']
        sent = nlp(cs['sentence'])
        for root in [tok for tok in sent if tok.head == tok]:
            if root.pos_ != "VERB":
                continue
            if root.lemma_ in BAD_TRIGGERS or root.lemma_ in WEAK_TRIGGERS:
                continue

            # Build event phrase
            tokens = [root]
            for child in root.children:
                if child.dep_ in {"dobj", "prep", "prt"}:
                    tokens.extend(list(child.subtree))
            tokens = sorted(set(tokens), key=lambda t: t.i)
            event_phrase = " ".join(t.text for t in tokens)

            events.append({
                "trigger": root.lemma_,
                "event_phrase": event_phrase,
                "sentence": cs['sentence'],
                "sent_id": sent_id,
                "chapter_id": cs['chapter_id'],
                "chapter_title": cs['chapter_title']
            })
    return events

events = extract_events(chapter_sentences)
print("Extracted events:", len(events))


Extracted events: 948


In [7]:
EVENT_TYPES = [
    "motion",
    "perception",
    "action",
    "communication",
    "cognition",
    "emotion"
]

def classify_events(events):
    for e in events:
        result = classifier(e["event_phrase"], EVENT_TYPES)
        e["event_type"] = result["labels"][:3]
    return events

events = classify_events(events)


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [8]:
def embed_events(events):
    vectors = []
    for e in events:
        emb = embedder(e["event_phrase"])[0]
        emb = np.mean(emb, axis=0)
        vectors.append(emb)
    return normalize(np.array(vectors))

X = embed_events(events)

def cluster_events(X, k=6):
    kmeans = KMeans(n_clusters=k, random_state=42)
    return kmeans.fit_predict(X)

clusters = cluster_events(X, k=6)

for i, c in enumerate(clusters):
    events[i]["cluster"] = int(c)


In [9]:
chapter_narrative = defaultdict(list)
for e in events:
    chapter_narrative[e['chapter_id']].append({
        "sentence_id": e['sent_id'],
        "event_phrase": e['event_phrase'],
        "event_type": e['event_type'][0],
        "cluster": e['cluster']
    })

# Inspect CHAPTER I
for e in chapter_narrative[1][:20]:
    print(f"{e['sentence_id']} → {e['event_phrase']} | type={e['event_type']} | cluster={e['cluster']}")


6 → In another moment went after it | type=action | cluster=2
7 → went like a tunnel for some way | type=motion | cluster=2
9 → saw | type=motion | cluster=5
10 → labelled | type=action | cluster=4
13 → say anything about it | type=communication | cluster=4
16 → come to an end | type=motion | cluster=2
18 → said | type=communication | cluster=0
19 → getting | type=motion | cluster=5
27 → listening | type=communication | cluster=4
31 → see Down , down , down | type=motion | cluster=2
41 → tell the truth | type=communication | cluster=4
43 → came upon a heap of sticks and dry leaves | type=motion | cluster=2
45 → went like the wind | type=motion | cluster=2
51 → knelt down | type=action | cluster=2
56 → said | type=communication | cluster=0
58 → read several nice little histories about children who had got burnt | type=emotion | cluster=2
59 → finished it off | type=action | cluster=2
60 → said | type=communication | cluster=0
62 → said to herself in my going out altogether , like a cand

In [10]:
# Select main events per chapter
def get_main_events(chapter_narrative, top_n=5):
    main_events_per_chapter = {}
    for chap_id, events in chapter_narrative.items():
        cluster_counter = Counter()
        for e in events:
            key = (e["cluster"], e["event_phrase"])
            cluster_counter[key] += 1
        seen_clusters = set()
        main_events = []
        for (cluster, phrase), freq in cluster_counter.most_common():
            if cluster not in seen_clusters:
                main_events.append({"event_phrase": phrase, "event_type": [e["event_type"] for e in events if e["event_phrase"] == phrase][0], "cluster": cluster, "frequency": freq})
                seen_clusters.add(cluster)
            if len(main_events) >= top_n:
                break
        main_events_per_chapter[chap_id] = main_events
    return main_events_per_chapter

main_events = get_main_events(chapter_narrative, top_n=5)

# Map main events to full sentences
def get_main_events_sentences(main_events, chapter_narrative, chapter_sentences):
    main_events_full = {}
    sent_lookup = {cs["global_sent_id"]: cs["sentence"] for cs in chapter_sentences}
    for chap_id, events in main_events.items():
        main_events_full[chap_id] = []
        for e in events:
            for ev in chapter_narrative[chap_id]:
                if ev["event_phrase"] == e["event_phrase"]:
                    full_sentence = sent_lookup.get(ev["sentence_id"], "")
                    main_events_full[chap_id].append({
                        "sentence": full_sentence,
                        "event_phrase": e["event_phrase"],
                        "event_type": e["event_type"],
                        "cluster": e["cluster"],
                        "frequency": e["frequency"]
                    })
                    break
    return main_events_full

main_events_full = get_main_events_sentences(main_events, chapter_narrative, chapter_sentences)

# Build readable plot summary
def build_readable_plot(main_events_full, chapters):
    plot_summary = []
    for chap_id, events in main_events_full.items():
        chapter_title = chapters[chap_id]["title"]
        plot_summary.append(f"\n=== {chapter_title} ===\n")
        for e in events:
            plot_summary.append(f"- {e['sentence']}")
    return plot_summary

plot_summary = build_readable_plot(main_events_full, chapters)

# Print first chapter
for line in plot_summary[:100]:
    print(line)


=== CHAPTER I.Down the Rabbit-Hole ===

- she said aloud.
- In another moment down went Alice after it, never once considering how
in the world she was to get out again.
- First, she tried to look down and make out
what she was coming to, but it was too dark to see anything; then she
looked at the sides of the well, and noticed that they were filled with
cupboards and book-shelves; here and there she saw maps and pictures
hung upon pegs.
- She took down a jar from one of the shelves as she
passed; it was labelled “ORANGE MARMALADE”, but to her great
disappointment it was empty: she did not like to drop the jar for fear
of killing somebody underneath, so managed to put it into one of the
cupboards as she fell past it.
- First, however, she
waited for a few minutes to see if she was going to shrink any further:
she felt a little nervous about this; “for it might end, you know,”
said Alice to herself, “in my going out altogether, like a candle.

=== CHAPTER II.The Pool of Tears ===

- “Y

Summarizing the events to be read

In [11]:
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

Device set to use cuda:0


In [12]:
def summarize_sentence(sentence, max_len=20):
    if len(sentence.split()) < 12:
        return sentence  # already short
    summary = summarizer(
        sentence,
        max_length=max_len,
        min_length=5,
        do_sample=False
    )[0]["summary_text"]
    return summary

In [13]:
def summarize_main_events(main_events_full):
    summarized = {}
    for chap_id, events in main_events_full.items():
        summarized[chap_id] = []
        for e in events:
            short_phrase = summarize_sentence(e["sentence"])
            summarized[chap_id].append({
                "summary": short_phrase,
                "original_sentence": e["sentence"],
                "event_type": e["event_type"],
                "cluster": e["cluster"]
            })
    return summarized

summarized_events = summarize_main_events(main_events_full)

Your max_length is set to 20, but your input_length is only 15. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=7)
Your max_length is set to 20, but your input_length is only 18. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=9)


In [14]:
def build_plot_outline(summarized_events, chapters):
    for chap_id, events in summarized_events.items():
        print(f"\n=== {chapters[chap_id]['title']} ===")
        for i, e in enumerate(events, 1):
            print(f"{i}. {e['summary']}")

In [15]:
build_plot_outline(summarized_events, chapters)


=== CHAPTER I.Down the Rabbit-Hole ===
1. she said aloud.
2. "I never once considering howin the world she was to get out again. In
3. She tried to look down and make out what she was coming to, but it was
4. . She took down a jar from one of the shelves as she passed.
5. “For it might end, you know,” said Alice to herself,

=== CHAPTER II.The Pool of Tears ===
1. “You ought to be ashamed of yourself,” said Alice, “
2. She cried until there was a large pool all round her, about four inches
3. "I shall be a great deal too far off to troublemyself about you,"
4. She went on planning to herself how she would manage it. And she went
5. And how odd the directions will look!

     _

=== CHAPTER III.A Caucus-Race and a Long Tale ===
1. “Ugh!” said the Lory, with a shiver.
2. CNN.com will feature iReporter photos in a weekly Travel Snapshots gallery.
3. The Mouse, who seemed to be a person of authority among them,called out,
4. Alice kept her eyes anxiously fixed on it, for she felt sure she

In [28]:
import re
import numpy as np
import spacy
import nltk
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download("punkt")

# -----------------------------
# 1. Clean Project Gutenberg text
# -----------------------------
def clean_gutenberg(text):
    start = text.find("*** START")
    end = text.find("*** END")
    if start != -1 and end != -1:
        return text[start:end]
    return text


# -----------------------------
# 2. Split text into chapters
# -----------------------------
def split_into_chapters(text):
    chapters = {}
    current_chapter = None
    buffer = []

    for line in text.split("\n"):
        line = line.strip()
        if re.match(r"CHAPTER\s+[IVXL]+", line):
            if current_chapter:
                chapters[current_chapter] = " ".join(buffer)
            current_chapter = line
            buffer = []
        else:
            buffer.append(line)

    if current_chapter:
        chapters[current_chapter] = " ".join(buffer)

    return chapters


# -----------------------------
# 3. Sentence segmentation
# -----------------------------
nlp = spacy.load("en_core_web_sm")
def is_event_sentence(sent):
    doc = nlp(sent)

    has_verb = any(tok.pos_ == "VERB" for tok in doc)
    has_named_entity = len(doc.ents) > 0
    is_dialogue = sent.strip().startswith(("“", '"'))

    return has_verb and (has_named_entity or not is_dialogue)
def get_sentences(text):
    doc = nlp(text)
    sentences = []

    for sent in doc.sents:
        s = sent.text.strip()
        if len(s) > 20 and is_event_sentence(s):
            sentences.append(s)

    return sentences


# -----------------------------
# 4. Extract main events using BERT
# -----------------------------
def extract_main_events(sentences, top_k=5):
    model = SentenceTransformer("all-MiniLM-L6-v2")

    embeddings = model.encode(sentences)
    chapter_vector = np.mean(embeddings, axis=0)

    scores = cosine_similarity(embeddings, chapter_vector.reshape(1, -1)).flatten()
    top_indices = np.argsort(scores)[::-1][:top_k]

    main_events = [sentences[i] for i in top_indices]

    # keep original narrative order
    main_events = sorted(main_events, key=lambda s: sentences.index(s))
    return main_events


# -----------------------------
# 5. Run pipeline
# -----------------------------
if __name__ == "__main__":

    # Load Alice in Wonderland text
    with open("alice_clean.txt", "r", encoding="utf-8") as f:
        raw_text = f.read()

    clean_text = clean_gutenberg(raw_text)
    chapters = split_into_chapters(clean_text)

    story_plot = {}

    for chapter, chapter_text in chapters.items():
        sentences = get_sentences(chapter_text)

        if len(sentences) < 5:
            continue

        main_events = extract_main_events(sentences, top_k=5)
        story_plot[chapter] = main_events
        import json

        # -----------------------------
        # 6. Save the extracted events
        # -----------------------------
        with open("story_plot.json", "w", encoding="utf-8") as f:
            json.dump(story_plot, f, ensure_ascii=False, indent=2)

        print("✅ SBERT-selected events saved to story_plot.json")

    # -----------------------------
    # 6. Print readable plot
    # -----------------------------
    for chapter, events in story_plot.items():
        print("\n" + "=" * 100)
        print(chapter)
        print("=" * 100)
        for e in events:
            print("- " + e)



[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sibic\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


✅ SBERT-selected events saved to story_plot.json
✅ SBERT-selected events saved to story_plot.json
✅ SBERT-selected events saved to story_plot.json
✅ SBERT-selected events saved to story_plot.json
✅ SBERT-selected events saved to story_plot.json
✅ SBERT-selected events saved to story_plot.json
✅ SBERT-selected events saved to story_plot.json
✅ SBERT-selected events saved to story_plot.json
✅ SBERT-selected events saved to story_plot.json
✅ SBERT-selected events saved to story_plot.json
✅ SBERT-selected events saved to story_plot.json
✅ SBERT-selected events saved to story_plot.json

CHAPTER I.Down the Rabbit-Hole
- “Well!” thought Alice to herself, “after such a fall as this, I shall think nothing of tumbling down stairs!
- There were doors all round the hall, but they were all locked; and when Alice had been all the way down one side and up the other, trying every door, she walked sadly down the middle, wondering how she was ever to get out again.
- How she longed to get out of that da

In [24]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# -----------------------------
# 6. Load Qwen LLM (for rephrasing)
# -----------------------------
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"  # small + fast + good

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:51<00:00, 25.86s/it]
Some parameters are on the meta device because they were offloaded to the disk and cpu.


In [ ]:


# -----------------------------
# 2. Function to rephrase events
# -----------------------------
def rephrase_event_llm(sentence):
    prompt = f"""
Rewrite the following literary sentence as a single short, concrete plot event.
Rules:
- Use a clear subject and verb
- Be factual and neutral
- Do NOT add new information
- Do NOT use dialogue or quotes

Sentence:
{sentence}

Rewritten event:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        output = llm.generate(
            **inputs,
            max_new_tokens=40,
            temperature=0.2,
            do_sample=False
        )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    return text.split("Rewritten event:")[-1].strip()

# -----------------------------
# 3. Load the story_plot.json
# -----------------------------
with open("story_plot.json", "r", encoding="utf-8") as f:
    story_plot = json.load(f)

# -----------------------------
# 4. Rewrite all events
# -----------------------------
rewritten_plot = {}

for chapter, events in story_plot.items():
    rewritten_plot[chapter] = []
    for e in events:
        rewritten_event = rephrase_event_llm(e)
        rewritten_plot[chapter].append(rewritten_event)

# -----------------------------
# 5. Save rewritten plot
# -----------------------------
with open("rewritten_plot.json", "w", encoding="utf-8") as f:
    json.dump(rewritten_plot, f, ensure_ascii=False, indent=2)

# -----------------------------
# 6. Print rewritten plot
# -----------------------------
for chapter, events in rewritten_plot.items():
    print(f"\n📘 {chapter}")
    for i, event in enumerate(events, 1):
        print(f"{i}. {event}")